# Temporal learning from pixels and numerical sensors

Render a Gaussian marker at positions around a circle as 16 x 16 grayscale images. A separate nonlinear sensor observes the same latent states. Train a tiny CNN and an MLP independently on temporal pairs, then compare their geometries using fresh paired observations.

Local and scrambled dynamics preserve the stationary image/sensor distribution and the complete transition spectrum. This checks whether a finite-budget finding persists across two different observation types and encoder architectures. Synthetic marker images are not real-world validation.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_URL = 'https://github.com/AlexM3020/The-Role-of-Dynamical-Structure-in-Cross-Modal-Representation-Learning.git'
REPO_NAME = 'The-Role-of-Dynamical-Structure-in-Cross-Modal-Representation-Learning'
REPO_REF = 'main'  # Use an immutable commit when collecting final outputs.
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / REPO_NAME]
ROOT = next((p for p in candidates if (p / 'src/dynalign/worlds.py').exists()), None)
if ROOT is None:
    ROOT = Path.cwd() / REPO_NAME
    subprocess.check_call(['git', 'clone', REPO_URL, str(ROOT)])
    subprocess.check_call(['git', '-C', str(ROOT), 'checkout', REPO_REF])
sys.path.insert(0, str(ROOT / 'src'))
try:
    import numpy, scipy, pandas, matplotlib, torch
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ROOT)])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dynalign.experiments import environment, grid, run_grid
from dynalign.reporting import contrast_summary, plot_results
from dynalign.worlds import image_table, sensor_table, observe
print(environment())

## Configuration and data preview
No pretrained weights or external dataset are required. The CNN is held identical across dynamics conditions, as is the MLP. Their parameter counts need not match each other because comparisons are within an architecture pair; exact counts are reported. Noise is fixed at 0.03 for pixels and 0.05 for the standardized numerical sensor.

Start with `smoke`; benchmark before expanding to `pilot` or `full`. Save outputs on persistent storage when using a cloud VM.

In [ ]:
PROFILE = os.environ.get('DYNALIGN_PROFILE', 'smoke')
DEVICE = os.environ.get('DYNALIGN_DEVICE', 'auto')
base = Path('/kaggle/working') if Path('/kaggle/working').exists() else ROOT
OUTPUT_DIR = Path(os.environ.get('DYNALIGN_OUTPUT_DIR', str(base / 'outputs'))) / '04_pixels_and_sensors'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
specs = grid('pixels', PROFILE)
display(pd.DataFrame(specs))
images = image_table(32)
indices = np.arange(0, 32, 4)
noisy_images = observe(images, indices, 0.03, 1)
fig, axes = plt.subplots(1, len(indices), figsize=(12, 2))
for ax, state, image in zip(axes, indices, noisy_images):
    ax.imshow(image[0], cmap='gray', vmin=0, vmax=1); ax.set_title(f'State {state}'); ax.axis('off')
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'observation_preview.png', dpi=150)
plt.show()

## Train separately and evaluate together
Both encoders use the same temporal VICReg-style loss but separate observation draws, initializations, minibatch random generators, and optimizers. The renderer is not passed to either network. Latent-state labels and same-state image/sensor pairs enter only the frozen evaluation.

The output includes random-initialization similarity, because smooth sensors and convolutional bias can already induce shared geometry before learning.

In [ ]:
results = run_grid(specs, OUTPUT_DIR, DEVICE)
display(results[['condition', 'n_pairs', 'world', 'repeat', 'parameters_a', 'parameters_b', 'training_seconds', 'initial_cka', 'cka', 'cka_gain', 'any_collapse']])

## Inspect the complete pattern
The primary comparison is local minus scrambled at a fixed data budget. Procrustes is fitted on separate calibration observations, using only centering, scalar normalization, and one orthogonal map. State-centroid RSA averages sensor noise with known state labels and is a complementary diagnostic.

Do not claim transfer to natural images, audio, language, or unseen dynamics. Changes in finite-network learning difficulty are part of the locality intervention, not evidence for a new physical force in optimization.

In [ ]:
contrasts = contrast_summary(results, 'cka')
contrasts.to_csv(OUTPUT_DIR / 'paired_contrasts.csv', index=False)
display(contrasts)
display(results[['condition', 'n_pairs', 'centroid_rsa', 'procrustes_error', 'coordinate_r2_a', 'coordinate_r2_b', 'spectral_r2_a', 'spectral_r2_b', 'effective_rank_a', 'effective_rank_b', 'any_collapse']])
figure = plot_results(results, 'cka')
figure.savefig(OUTPUT_DIR / 'pixel_sensor_alignment.png', dpi=150)
plt.show()

## Preserve the run
The archive contains all saved configurations, checkpoints, embeddings, and tables. A completed configuration is cached, while interrupted training resumes from the latest matching checkpoint. Copy the archive to durable storage before ending the session.

In [ ]:
import shutil
from IPython.display import FileLink
archive = shutil.make_archive(str(OUTPUT_DIR), 'zip', OUTPUT_DIR)
print('Saved archive:', archive)
display(FileLink(archive))